# Step 1: 데이터 준비

KoDF(한국인 딥페이크) 샘플 데이터를 준비하고 S3에 업로드합니다.

## 실습 목표
- 딥페이크 탐지용 데이터셋 구조 이해
- Train/Val/Test 데이터 분리
- S3 버킷에 데이터 업로드

## 1.1 환경 설정

In [ ]:
import os
import boto3
import sagemaker
from sagemaker import get_execution_role

# SageMaker 세션 설정
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name

# S3 버킷 설정 (기본 버킷 사용 또는 직접 지정)
bucket = sagemaker_session.default_bucket()
prefix = 'deepfake-detection'

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Prefix: {prefix}")

## 1.2 샘플 데이터 다운로드

실습을 위해 공개된 딥페이크 샘플 데이터를 사용합니다.

> **참고**: 실제 KoDF 데이터는 [AI Hub](https://www.aihub.or.kr/)에서 신청 후 다운로드 가능합니다.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

# 데이터 디렉토리 생성
data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

# 샘플 데이터 구조 생성 (실제로는 KoDF 데이터 사용)
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        (data_dir / split / label).mkdir(parents=True, exist_ok=True)

print("데이터 디렉토리 구조:")
!find ./data -type d

## 1.3 데이터 구조 설명

```
data/
├── train/           # 학습 데이터 (70%)
│   ├── real/        # 진짜 영상 프레임
│   └── fake/        # 가짜 영상 프레임
├── val/             # 검증 데이터 (15%)
│   ├── real/
│   └── fake/
└── test/            # 테스트 데이터 (15%)
    ├── real/
    └── fake/
```

In [ ]:
# 샘플 이미지 생성 (데모용 - 실제로는 KoDF 프레임 사용)
import numpy as np
from PIL import Image
import random

def create_sample_images(data_dir, num_samples={'train': 100, 'val': 20, 'test': 20}):
    """
    데모용 샘플 이미지 생성
    실제 실습에서는 KoDF 데이터로 대체
    """
    for split, count in num_samples.items():
        for label in ['real', 'fake']:
            for i in range(count // 2):
                # 224x224 랜덤 이미지 생성
                img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
                img = Image.fromarray(img_array)
                img.save(data_dir / split / label / f'{label}_{i:04d}.jpg')
    
    print("샘플 이미지 생성 완료!")

# 샘플 데이터 생성
create_sample_images(data_dir)

# 데이터 개수 확인
for split in ['train', 'val', 'test']:
    real_count = len(list((data_dir / split / 'real').glob('*.jpg')))
    fake_count = len(list((data_dir / split / 'fake').glob('*.jpg')))
    print(f"{split}: Real={real_count}, Fake={fake_count}")

## 1.4 실제 KoDF 데이터 사용 시 (참고)

실제 KoDF 데이터를 사용할 경우 아래 코드로 비디오에서 프레임을 추출합니다.

In [ ]:
# 비디오에서 프레임 추출 함수 (참고용)
import cv2
from facenet_pytorch import MTCNN
import torch

def extract_faces_from_video(video_path, output_dir, num_frames=10):
    """
    비디오에서 얼굴 프레임 추출
    
    Args:
        video_path: 비디오 파일 경로
        output_dir: 출력 디렉토리
        num_frames: 추출할 프레임 수
    """
    # MTCNN 얼굴 검출기
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mtcnn = MTCNN(device=device, select_largest=True)
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    saved_count = 0
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
            
        # BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 얼굴 검출 및 크롭
        face = mtcnn(frame_rgb, save_path=None)
        if face is not None:
            # 텐서를 이미지로 변환하여 저장
            face_img = face.permute(1, 2, 0).cpu().numpy()
            face_img = ((face_img + 1) / 2 * 255).astype(np.uint8)
            img = Image.fromarray(face_img)
            img = img.resize((224, 224))
            img.save(output_dir / f'frame_{saved_count:04d}.jpg')
            saved_count += 1
    
    cap.release()
    return saved_count

print("얼굴 추출 함수 정의 완료 (KoDF 데이터 사용 시 활용)")

## 1.5 S3 업로드

In [ ]:
# S3에 데이터 업로드
s3_data_path = sagemaker_session.upload_data(
    path='./data',
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"데이터가 업로드되었습니다: {s3_data_path}")

In [ ]:
# S3 경로 저장 (다음 노트북에서 사용)
import json

config = {
    'bucket': bucket,
    'prefix': prefix,
    's3_data_path': s3_data_path,
    's3_train_path': f's3://{bucket}/{prefix}/data/train',
    's3_val_path': f's3://{bucket}/{prefix}/data/val',
    's3_test_path': f's3://{bucket}/{prefix}/data/test',
    'role': role,
    'region': region
}

with open('../config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("설정이 저장되었습니다: ../config.json")
print(json.dumps(config, indent=2))

## 완료!

데이터 준비가 완료되었습니다. 다음 단계로 이동하세요:

**➡️ `2_before_evaluation/evaluate_before.ipynb`**